# Locy Neural Use Case: Adverse Drug Reaction Signal Detection (Rust)

Compact Rust counterpart to the Python ADR flagship. Same shape: an inline pharmacovigilance graph + a `MockClassifier` registered under the model name + `CREATE MODEL` + `CALIBRATE` + `VALIDATE` + `EXPLAIN`.


## 1) Setup + Schema


In [ ]:
use std::sync::Arc;
use uni_db::{DataType, Uni, Result};
use uni_locy::{LocyConfig, MockClassifier, NeuralClassifier, FeatureValue};

let db = Uni::in_memory().build().await?;
db.schema()
    .label("Report")
        .property("report_id", DataType::String)
        .property("report_count", DataType::Float64)
        .property("is_signal", DataType::Bool)
    .done()
    .apply()
    .await?;


## 2) Seed 12 Reports (4 Signals)


In [ ]:
let session = db.session();
let tx = session.tx().await?;
let rows: &[(&str, f64, bool)] = &[
    ("R01", 9.2, true),  ("R02", 2.0, false),
    ("R03", 8.7, true),  ("R04", 1.6, false),
    ("R05", 2.4, false), ("R06", 3.0, false),
    ("R07", 9.5, true),  ("R08", 2.2, false),
    ("R09", 8.9, true),  ("R10", 2.6, false),
    ("R11", 3.1, false), ("R12", 2.0, false),
];
for (rid, rc, is_signal) in rows {
    let q = format!(
        "CREATE (:Report {{report_id: '{}', report_count: {}, is_signal: {}}})",
        rid, rc, is_signal
    );
    tx.execute(&q).await?;
}
tx.commit().await?;


## 3) Register the Signal Classifier


In [ ]:
let classifier: Arc<dyn NeuralClassifier> = Arc::new(
    MockClassifier::new("signal_score", |inp| {
        let rc = match inp.features.get("r") {
            Some(FeatureValue::Float(v)) => *v,
            _ => 0.0,
        };
        let z = (rc - 5.0) * 0.9 - 0.5;
        let p = 1.0 / (1.0 + (-z).exp());
        let p_sharp = 1.0 / (1.0 + (-3.5 * (p - 0.5)).exp());
        p_sharp.clamp(0.0, 1.0)
    })
);
let mut config = LocyConfig::default();
config.classifier_registry.insert("signal_score".to_string(), classifier);


## 4) CREATE MODEL + Score


In [ ]:
let program = r#"
CREATE MODEL signal_score AS
  INPUT (r)
  FEATURES r.report_count
  OUTPUT PROB credibility
  USING xervo('classify/adr-signal-v1')

CREATE RULE scored_reports AS
  MATCH (r:Report)
  YIELD KEY r, signal_score(r.report_count) AS credibility
"#;
let result = session.locy_with(program).with_config(config.clone()).run().await?;
let n = result.derived().get("scored_reports").map(|v| v.len()).unwrap_or(0);
println!("Scored {} reports", n);


## 5) CALIBRATE Against Held-Out Signal Labels


In [ ]:
let calibrate_program = r#"
CREATE MODEL signal_score AS
  INPUT (r)
  FEATURES r.report_count
  OUTPUT PROB credibility
  USING xervo('classify/adr-signal-v1')

CALIBRATE signal_score
  ON MATCH (r:Report)
  TARGET r.is_signal
  METHOD platt_scaling
"#;
let calib = session.locy_with(calibrate_program).with_config(config.clone()).run().await?;
println!("command_results: {} entries", calib.command_results().len());


## 6) EXPLAIN One High-Credibility Report


In [ ]:
let explain_program = format!("{}{}", program, r#"

EXPLAIN RULE scored_reports WHERE r.report_id = 'R01'
"#);
let explain = session.locy_with(&explain_program).with_config(config).run().await?;
println!("EXPLAIN command_results: {}", explain.command_results().len());
